# 02 - Learning to Rank

Este notebook construye el dataset de ranking y entrena un `XGBRanker`. La unidad de ranking es una orden objetivo: para cada orden se arma una lista de productos candidatos y el modelo aprende a ordenarlos.

Para mantener el notebook manejable durante la prueba, trabajo con una muestra de usuarios. El flujo completo queda igual si se aumenta el tamaño de la muestra.


## Carga de datos procesados

Busco automáticamente los archivos Parquet generados en el notebook anterior. Así el notebook no depende de una ruta escrita a mano.


In [2]:
from pathlib import Path

INPUT = Path("/kaggle/input")

parquet_files = list(INPUT.rglob("prior_items.parquet"))

if not parquet_files:
    raise FileNotFoundError("No se encontró prior_items.parquet. Agrega como input el output del notebook 01_prepare_data.")

DATA_DIR = parquet_files[0].parent

print("Usando datos procesados desde:")
print(DATA_DIR)

print("\nArchivos encontrados:")
for file in DATA_DIR.iterdir():
    print(file.name)

Usando datos procesados desde:
/kaggle/input/notebooks/samanthacarolina/01-prepare-data

Archivos encontrados:
__results__.html
product_catalog.parquet
__notebook__.ipynb
orders.parquet
__output__.json
train_items.parquet
prior_items.parquet
custom.css


## Lectura de Parquets

Cargo las tablas ya procesadas: órdenes, historial de compras, última orden etiquetada y catálogo.


In [3]:
import pandas as pd
from pathlib import Path

orders = pd.read_parquet(DATA_DIR / "orders.parquet")
prior_items = pd.read_parquet(DATA_DIR / "prior_items.parquet")
train_items = pd.read_parquet(DATA_DIR / "train_items.parquet")
product_catalog = pd.read_parquet(DATA_DIR / "product_catalog.parquet")

print("orders:", orders.shape)
print("prior_items:", prior_items.shape)
print("train_items:", train_items.shape)
print("product_catalog:", product_catalog.shape)

orders: (3421083, 7)
prior_items: (32434489, 14)
train_items: (1384617, 14)
product_catalog: (49688, 6)


## Muestra de usuarios

Uso una muestra de usuarios que sí tienen orden final en `train`. Esto permite entrenar y evaluar más rápido sin cambiar la lógica del problema.


In [4]:
SAMPLE_USERS = 5000

# Mejor: samplear solo usuarios que sí tengan orden final en train
available_train_users = train_items["user_id"].drop_duplicates()

sample_user_ids = available_train_users.sample(
    n=min(SAMPLE_USERS, len(available_train_users)),
    random_state=42
)

orders_s = orders[orders["user_id"].isin(sample_user_ids)].copy()
prior_s = prior_items[prior_items["user_id"].isin(sample_user_ids)].copy()
train_s = train_items[train_items["user_id"].isin(sample_user_ids)].copy()

print("orders_s:", orders_s.shape)
print("prior_s:", prior_s.shape)
print("train_s:", train_s.shape)
print("Usuarios con train:", train_s["user_id"].nunique())

orders_s: (82549, 7)
prior_s: (789017, 14)
train_s: (53018, 14)
Usuarios con train: 5000


## Verificación de la muestra

Reviso cuántos usuarios quedaron con historial y cuántos tienen datos en `train`. Si `train_s` quedara vacío, la evaluación final no serviría.


In [5]:
print("Usuarios sampleados:", len(sample_user_ids))
print("Usuarios con prior:", prior_s["user_id"].nunique())
print("Usuarios con train:", train_s["user_id"].nunique())
print("Filas train_s:", train_s.shape)

Usuarios sampleados: 5000
Usuarios con prior: 5000
Usuarios con train: 5000
Filas train_s: (53018, 14)


## Target de entrenamiento

Para entrenar uso la última orden `prior` de cada usuario como orden objetivo. Las órdenes anteriores forman el historial usado para calcular features.


In [6]:
# Para entrenamiento:
# Usaremos la última orden prior de cada usuario como orden objetivo.
# El historial serán las órdenes prior anteriores.

user_last_prior = (
    prior_s
    .groupby("user_id")["order_number"]
    .max()
    .reset_index(name="target_order_number")
)

# Quitamos usuarios con una sola orden prior porque no tendrían historial anterior
eligible_train_users = user_last_prior[
    user_last_prior["target_order_number"] > 1
].copy()

prior_with_target = prior_s.merge(
    eligible_train_users,
    on="user_id",
    how="inner"
)

train_target_items = prior_with_target[
    prior_with_target["order_number"] == prior_with_target["target_order_number"]
].copy()

train_history = prior_with_target[
    prior_with_target["order_number"] < prior_with_target["target_order_number"]
].copy()

print("train_target_items:", train_target_items.shape)
print("train_history:", train_history.shape)
print("Órdenes objetivo train:", train_target_items["order_id"].nunique())
print("Usuarios train:", train_target_items["user_id"].nunique())

train_target_items: (52134, 15)
train_history: (736883, 15)
Órdenes objetivo train: 5000
Usuarios train: 5000


## Target de evaluación

Para test uso `order_products__train.csv`, que representa la última orden real etiquetada. En esta parte no se usa `train` para entrenar.


In [7]:
# Para test:
# Historial = todo el prior del usuario
# Target = productos reales de order_products__train

test_target_items = train_s.copy()

test_history = prior_s[
    prior_s["user_id"].isin(test_target_items["user_id"].unique())
].copy()

print("test_target_items:", test_target_items.shape)
print("test_history:", test_history.shape)
print("Órdenes objetivo test:", test_target_items["order_id"].nunique())
print("Usuarios test:", test_target_items["user_id"].nunique())

test_target_items: (53018, 14)
test_history: (789017, 14)
Órdenes objetivo test: 5000
Usuarios test: 5000


## Labels positivos

Los productos comprados en la orden objetivo reciben label 1 o 2. Uso label 2 cuando el producto fue reordenado y label 1 cuando aparece como compra nueva.


In [8]:
import numpy as np
import pandas as pd

def make_positive_rows(target_items: pd.DataFrame) -> pd.DataFrame:
    """
    Crea filas positivas:
    label = 2 si reordered = 1
    label = 1 si reordered = 0
    """
    positives = target_items[[
        "order_id",
        "user_id",
        "product_id",
        "reordered"
    ]].copy()

    positives = positives.rename(columns={"order_id": "qid"})

    positives["label"] = np.where(
        positives["reordered"] == 1,
        2,
        1
    )

    positives = positives[[
        "qid",
        "user_id",
        "product_id",
        "label"
    ]]

    return positives

## Revisión de positivos

Genero los positivos para entrenamiento y test, y reviso la distribución de labels.


In [9]:
train_pos = make_positive_rows(train_target_items)
test_pos = make_positive_rows(test_target_items)

print("train_pos:", train_pos.shape)
print("test_pos:", test_pos.shape)

print("\nLabels train:")
print(train_pos["label"].value_counts())

print("\nLabels test:")
print(test_pos["label"].value_counts())

train_pos.head()

train_pos: (52134, 4)
test_pos: (53018, 4)

Labels train:
label
2    30623
1    21511
Name: count, dtype: int64

Labels test:
label
2    31765
1    21253
Name: count, dtype: int64


,qid,user_id,product_id,label
37,198,21336,33000,2
38,198,21336,32374,2
39,198,21336,2421,2
40,198,21336,20842,2
41,198,21336,24852,2


## Catálogo para negative sampling

Preparo un catálogo mínimo por producto y departamento. Esto permite samplear negativos del mismo departamento que los positivos.


In [10]:
catalog_min = product_catalog[[
    "product_id",
    "product_name",
    "department_id",
    "department",
    "aisle"
]].drop_duplicates("product_id").copy()

products_by_department = (
    catalog_min
    .groupby("department_id")["product_id"]
    .apply(lambda x: set(x.tolist()))
    .to_dict()
)

print("Productos catálogo:", catalog_min.shape)
print("Departamentos:", len(products_by_department))
catalog_min.head()

Productos catálogo: (49688, 5)
Departamentos: 21


,product_id,product_name,department_id,department,aisle
0,1,Chocolate Sandwich Cookies,19,snacks,cookies cakes
1,2,All-Seasons Salt,13,pantry,spices seasonings
2,3,Robust Golden Unsweetened Oolong Tea,7,beverages,tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,1,frozen,frozen meals
4,5,Green Chile Anytime Sauce,13,pantry,marinades meat preparation


## Negative sampling

La prueba pide negativos del mismo departamento y que el usuario no haya comprado antes. La función también excluye productos que sí aparecen en la orden objetivo para no marcar como negativo algo que en realidad fue comprado.


In [11]:
def sample_negatives(
    positives: pd.DataFrame,
    history: pd.DataFrame,
    catalog: pd.DataFrame,
    products_by_department: dict,
    negatives_per_positive: int = 5,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Genera negativos del mismo departamento que el positivo.

    Reglas:
    - Mismo department_id que un producto positivo.
    - Producto que el usuario nunca compró en su historial.
    - Producto que no aparece en la orden objetivo.
    - label = 0
    """

    local_rng = np.random.default_rng(random_state)

    positives_dept = positives.merge(
        catalog[["product_id", "department_id"]],
        on="product_id",
        how="left"
    )

    history_products_by_user = (
        history
        .groupby("user_id")["product_id"]
        .apply(lambda x: set(x.tolist()))
        .to_dict()
    )

    target_products_by_qid = (
        positives
        .groupby("qid")["product_id"]
        .apply(lambda x: set(x.tolist()))
        .to_dict()
    )

    negatives = []
    selected_negatives_by_qid = {}

    for row in positives_dept.itertuples(index=False):
        qid = row.qid
        user_id = row.user_id
        department_id = row.department_id

        if pd.isna(department_id):
            continue

        department_id = int(department_id)

        dept_products = products_by_department.get(department_id, set())
        user_history_products = history_products_by_user.get(user_id, set())
        target_products = target_products_by_qid.get(qid, set())
        already_selected = selected_negatives_by_qid.setdefault(qid, set())

        eligible = (
            dept_products
            - user_history_products
            - target_products
            - already_selected
        )

        if not eligible:
            continue

        n_to_sample = min(negatives_per_positive, len(eligible))

        sampled = local_rng.choice(
            list(eligible),
            size=n_to_sample,
            replace=False
        )

        for neg_product_id in sampled:
            negatives.append({
                "qid": qid,
                "user_id": user_id,
                "product_id": int(neg_product_id),
                "label": 0
            })

        already_selected.update(set(sampled.tolist()))

    return pd.DataFrame(negatives)

## Generación de negativos

Uso 5 negativos por positivo en entrenamiento y 20 en test. En test uso más negativos para que la evaluación sea un poco más exigente.


In [12]:
NEG_PER_POS_TRAIN = 5
NEG_PER_POS_TEST = 20
RANDOM_STATE = 42

train_neg = sample_negatives(
    positives=train_pos,
    history=train_history,
    catalog=catalog_min,
    products_by_department=products_by_department,
    negatives_per_positive=NEG_PER_POS_TRAIN,
    random_state=RANDOM_STATE
)

test_neg = sample_negatives(
    positives=test_pos,
    history=test_history,
    catalog=catalog_min,
    products_by_department=products_by_department,
    negatives_per_positive=NEG_PER_POS_TEST,
    random_state=RANDOM_STATE + 1
)

print("train_neg:", train_neg.shape)
print("test_neg:", test_neg.shape)

train_neg.head()

train_neg: (260670, 4)
test_neg: (1060350, 4)


,qid,user_id,product_id,label
0,198,21336,47279,0
1,198,21336,19937,0
2,198,21336,29921,0
3,198,21336,738,0
4,198,21336,19879,0


## Dataset candidato

Uno positivos y negativos. A partir de aquí cada fila representa un par `(usuario, producto)` dentro de una query de ranking.


In [13]:
train_candidates = pd.concat(
    [train_pos, train_neg],
    ignore_index=True
)

test_candidates = pd.concat(
    [test_pos, test_neg],
    ignore_index=True
)

print("train_candidates:", train_candidates.shape)
print("test_candidates:", test_candidates.shape)

print("\nLabels train:")
print(train_candidates["label"].value_counts())

print("\nLabels test:")
print(test_candidates["label"].value_counts())

train_candidates.head()

train_candidates: (312804, 4)
test_candidates: (1113368, 4)

Labels train:
label
0    260670
2     30623
1     21511
Name: count, dtype: int64

Labels test:
label
0    1060350
2      31765
1      21253
Name: count, dtype: int64


,qid,user_id,product_id,label
0,198,21336,33000,2
1,198,21336,32374,2
2,198,21336,2421,2
3,198,21336,20842,2
4,198,21336,24852,2


## Estado del dataset base

Hasta aquí ya están los labels requeridos: 2 para reordenado, 1 para compra nueva y 0 para productos negativos.


In [14]:
def add_required_features(
    candidates: pd.DataFrame,
    history: pd.DataFrame,
    catalog: pd.DataFrame
) -> pd.DataFrame:
    """
    Agrega las 4 features requeridas por la prueba:
    - user_product_frequency
    - user_product_reordered
    - product_global_popularity
    - department_id
    """

    # 1 y 2: features usuario-producto
    user_product_features = (
        history
        .groupby(["user_id", "product_id"])
        .agg(
            user_product_frequency=("order_id", "nunique"),
            user_product_reordered=("reordered", "max")
        )
        .reset_index()
    )

    # 3: popularidad global del producto
    product_popularity = (
        history
        .groupby("product_id")["user_id"]
        .nunique()
        .reset_index(name="product_global_popularity")
    )

    # 4: department_id
    product_department = catalog[[
        "product_id",
        "department_id"
    ]].drop_duplicates("product_id")

    df = (
        candidates
        .merge(
            user_product_features,
            on=["user_id", "product_id"],
            how="left"
        )
        .merge(
            product_popularity,
            on="product_id",
            how="left"
        )
        .merge(
            product_department,
            on="product_id",
            how="left"
        )
    )

    # Los productos nuevos o negativos pueden no existir en el historial del usuario
    df["user_product_frequency"] = df["user_product_frequency"].fillna(0)
    df["user_product_reordered"] = df["user_product_reordered"].fillna(0)
    df["product_global_popularity"] = df["product_global_popularity"].fillna(0)
    df["department_id"] = df["department_id"].fillna(-1)

    # Tipos más livianos
    df["user_product_frequency"] = df["user_product_frequency"].astype("int32")
    df["user_product_reordered"] = df["user_product_reordered"].astype("int8")
    df["product_global_popularity"] = df["product_global_popularity"].astype("int32")
    df["department_id"] = df["department_id"].astype("int16")
    df["label"] = df["label"].astype("int8")

    return df

## Features obligatorias

Agrego las cuatro features pedidas en la prueba: frecuencia usuario-producto, si alguna vez fue reordenado, popularidad global y departamento.


In [15]:
train_df = add_required_features(
    candidates=train_candidates,
    history=train_history,
    catalog=catalog_min
)

test_df = add_required_features(
    candidates=test_candidates,
    history=test_history,
    catalog=catalog_min
)

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)

train_df.head()

train_df: (312804, 8)
test_df: (1113368, 8)


,qid,user_id,product_id,label,user_product_frequency,user_product_reordered,product_global_popularity,department_id
0,198,21336,33000,2,4,1,245,16
1,198,21336,32374,2,7,1,4,19
2,198,21336,2421,2,7,1,2,19
3,198,21336,20842,2,1,0,192,16
4,198,21336,24852,2,1,0,1731,4


## Construcción de features

Aplico las features al set de entrenamiento y al set de evaluación, usando el historial correspondiente en cada caso.


In [16]:
required_columns = [
    "qid",
    "user_id",
    "product_id",
    "user_product_frequency",
    "user_product_reordered",
    "product_global_popularity",
    "department_id",
    "label"
]

assert all(col in train_df.columns for col in required_columns)
assert all(col in test_df.columns for col in required_columns)

assert set(train_df["label"].unique()).issubset({0, 1, 2})
assert set(test_df["label"].unique()).issubset({0, 1, 2})

# Cada query debe tener al menos un producto positivo
train_positive_per_qid = train_df.groupby("qid")["label"].apply(lambda x: (x > 0).sum())
test_positive_per_qid = test_df.groupby("qid")["label"].apply(lambda x: (x > 0).sum())

assert (train_positive_per_qid > 0).all()
assert (test_positive_per_qid > 0).all()

print("Validaciones OK")

print("\nDistribución labels train:")
print(train_df["label"].value_counts())

print("\nDistribución labels test:")
print(test_df["label"].value_counts())

print("\nColumnas:")
print(train_df.columns.tolist())

Validaciones OK

Distribución labels train:
label
0    260670
2     30623
1     21511
Name: count, dtype: int64

Distribución labels test:
label
0    1060350
2      31765
1      21253
Name: count, dtype: int64

Columnas:
['qid', 'user_id', 'product_id', 'label', 'user_product_frequency', 'user_product_reordered', 'product_global_popularity', 'department_id']


## Validaciones básicas

Antes de entrenar reviso columnas, labels y que cada query tenga al menos un positivo. Esto evita entrenar con grupos mal formados.


In [17]:
def ndcg_at_k_single(y_true, y_score, k=10):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    order = np.argsort(y_score)[::-1][:k]
    ranked_labels = y_true[order]

    gains = (2 ** ranked_labels) - 1
    discounts = np.log2(np.arange(2, len(ranked_labels) + 2))

    dcg = np.sum(gains / discounts)

    ideal_order = np.argsort(y_true)[::-1][:k]
    ideal_labels = y_true[ideal_order]

    ideal_gains = (2 ** ideal_labels) - 1
    ideal_discounts = np.log2(np.arange(2, len(ideal_labels) + 2))

    idcg = np.sum(ideal_gains / ideal_discounts)

    if idcg == 0:
        return 0.0

    return float(dcg / idcg)


def average_precision_at_k_single(y_true, y_score, k=10):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Para MAP usamos relevancia binaria:
    # label 1 y 2 son relevantes; label 0 no relevante.
    relevant = (y_true > 0).astype(int)

    order = np.argsort(y_score)[::-1][:k]
    ranked_relevant = relevant[order]

    total_relevant = relevant.sum()

    if total_relevant == 0:
        return 0.0

    precisions = np.cumsum(ranked_relevant) / (
        np.arange(len(ranked_relevant)) + 1
    )

    ap = np.sum(precisions * ranked_relevant) / min(total_relevant, k)

    return float(ap)


def evaluate_ranking(df, score_col, k=10):
    ndcgs = []
    aps = []

    for _, group in df.groupby("qid"):
        y_true = group["label"].values
        y_score = group[score_col].values

        ndcgs.append(ndcg_at_k_single(y_true, y_score, k=k))
        aps.append(average_precision_at_k_single(y_true, y_score, k=k))

    return {
        f"ndcg_at_{k}": float(np.mean(ndcgs)),
        f"map_at_{k}": float(np.mean(aps)),
        "num_queries": int(df["qid"].nunique())
    }

## Métricas de ranking

Implemento NDCG@10 y MAP@10. Para NDCG uso labels graduados; para MAP convierto los labels 1 y 2 a relevancia binaria.


In [18]:
test_df["baseline_score"] = test_df["product_global_popularity"]

baseline_metrics = evaluate_ranking(
    test_df,
    score_col="baseline_score",
    k=10
)

baseline_metrics

{'ndcg_at_10': 0.5031041751533476,
 'map_at_10': 0.3679661195515243,
 'num_queries': 5000}

## Baseline de popularidad

El baseline ordena productos por popularidad global. La comparación se hace sobre el mismo conjunto de candidatos del modelo.


In [19]:
from xgboost import XGBRanker

## Import del ranker

Uso `XGBRanker` con objective `rank:ndcg`, que encaja con el objetivo de ordenar productos por relevancia.


In [20]:
features = [
    "user_product_frequency",
    "user_product_reordered",
    "product_global_popularity",
    "department_id"
]

# XGBRanker necesita que los datos estén ordenados por grupo/query
train_df = train_df.sort_values(["qid", "product_id"]).reset_index(drop=True)
test_df = test_df.sort_values(["qid", "product_id"]).reset_index(drop=True)

X_train = train_df[features]
y_train = train_df["label"]

group_train = (
    train_df
    .groupby("qid", sort=False)
    .size()
    .to_numpy()
)

assert group_train.sum() == len(train_df)

print("Filas train:", len(train_df))
print("Cantidad de grupos:", len(group_train))
print("Primeros tamaños de grupo:", group_train[:10])

Filas train: 312804
Cantidad de grupos: 5000
Primeros tamaños de grupo: [ 60  30  66  36 156  96  18 132 120  66]


## Preparación de grupos

El parámetro `group` es clave: cada grupo representa los candidatos de una orden objetivo (`qid`). Por eso ordeno el dataframe por `qid` antes de entrenar.


In [21]:
ranker = XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg@10",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)

print("Modelo entrenado correctamente")

Modelo entrenado correctamente


## Entrenamiento del modelo

Entreno el ranker con las cuatro features solicitadas. Mantengo parámetros simples para que el experimento sea reproducible y no dependa demasiado de tuning.


In [22]:
test_df["model_score"] = ranker.predict(test_df[features])

model_metrics = evaluate_ranking(
    test_df,
    score_col="model_score",
    k=10
)

print("Baseline:")
print(baseline_metrics)

print("\nXGBRanker:")
print(model_metrics)

Baseline:
{'ndcg_at_10': 0.5031041751533476, 'map_at_10': 0.3679661195515243, 'num_queries': 5000}

XGBRanker:
{'ndcg_at_10': 0.9187133675832192, 'map_at_10': 0.8016026298500881, 'num_queries': 5000}


## Evaluación del ranker

Predigo scores sobre el test y calculo las mismas métricas usadas para el baseline.


In [23]:
results_df = pd.DataFrame([
    {
        "model": "Global popularity baseline",
        "ndcg_at_10": baseline_metrics["ndcg_at_10"],
        "map_at_10": baseline_metrics["map_at_10"],
        "num_queries": baseline_metrics["num_queries"]
    },
    {
        "model": "XGBRanker",
        "ndcg_at_10": model_metrics["ndcg_at_10"],
        "map_at_10": model_metrics["map_at_10"],
        "num_queries": model_metrics["num_queries"]
    }
])

results_df

,model,ndcg_at_10,map_at_10,num_queries
0,Global popularity baseline,0.503104,0.367966,5000
1,XGBRanker,0.918713,0.801603,5000


## Comparación modelo vs baseline

Esta tabla resume la parte principal de la etapa de ranking: popularidad global contra XGBRanker.


In [24]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": ranker.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
1,user_product_reordered,0.502542
0,user_product_frequency,0.484581
2,product_global_popularity,0.011094
3,department_id,0.001782


## Importancia de features

Reviso qué variables está usando más el modelo. Esto ayuda a interpretar si el ranker se apoya en señales razonables.


In [25]:
import json
import joblib
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working")

ranking_report = {
    "sample_users": SAMPLE_USERS,
    "negative_sampling": {
        "train_negatives_per_positive": NEG_PER_POS_TRAIN,
        "test_negatives_per_positive": NEG_PER_POS_TEST,
        "strategy": "same_department_products_never_bought_by_user"
    },
    "features": features,
    "baseline": baseline_metrics,
    "xgb_ranker": model_metrics,
    "feature_importance": feature_importance.to_dict(orient="records")
}

# Guardar datasets LTR
train_df.to_parquet(OUTPUT_DIR / "ltr_train.parquet", index=False)
test_df.to_parquet(OUTPUT_DIR / "ltr_test.parquet", index=False)

# Guardar modelo
joblib.dump(ranker, OUTPUT_DIR / "xgb_ranker.joblib")

# Guardar métricas
with open(OUTPUT_DIR / "ranking_metrics.json", "w") as f:
    json.dump(ranking_report, f, indent=2)

print("Archivos guardados:")
print("- ltr_train.parquet")
print("- ltr_test.parquet")
print("- xgb_ranker.joblib")
print("- ranking_metrics.json")

Archivos guardados:
- ltr_train.parquet
- ltr_test.parquet
- xgb_ranker.joblib
- ranking_metrics.json


## Guardado de outputs

Guardo el modelo, los datasets LTR y el reporte de métricas. Estos archivos se usan después para crear los artefactos de la demo.


In [26]:
import os

os.listdir("/kaggle/working")

['__notebook__.ipynb',
 'ranking_metrics.json',
 'xgb_ranker.joblib',
 'ltr_test.parquet',
 'ltr_train.parquet']